# H₁₂ Hydrogen Chain — Ground State via KLT Solver

Maps the **12-site 1D antiferromagnetic Heisenberg chain** onto the KLT interaction matrix
and finds the ground-state spin configuration using Qumulator's KLT ground-state engine.

| Property | Value |
|----------|-------|
| Sites (qubits) | 12 |
| Coupling | $J_{i,i+1} = +1$ (antiferromagnetic) |
| Target energy | $E_0 = -(N-1) = -11$ (normalised) |
| Expected spin config | $\downarrow\uparrow\downarrow\uparrow\ldots$ (Néel state) |

**This is the same problem as a VQE ground-state search** — the KLT solver finds the
minimum-energy spin configuration without the variational ansatz overhead.

**Change your API key in the cell below, then run all cells.**

In [ ]:
# ── Set your API key ───────────────────────────────────────────────────────────
# Get a free key: curl -s -X POST https://api.qumulator.com/keys \
#   -H 'Content-Type: application/json' -d '{"name": "my-key"}' | python -m json.tool

import os
API_KEY = os.environ.get('QUMULATOR_API_KEY', 'YOUR_KEY_HERE')
API_URL = 'https://api.qumulator.com'

In [ ]:
%pip install qumulator-sdk --quiet
from qumulator import QumulatorClient
import numpy as np
import time

client = QumulatorClient(api_url=API_URL, api_key=API_KEY)
print('SDK ready.')

## Build the Heisenberg Hamiltonian

The 1D antiferromagnetic Heisenberg chain has nearest-neighbour coupling $J_{i,i+1} = +1$.
We pass this as a tridiagonal $12 \times 12$ interaction matrix to the KLT solver.

In [ ]:
N = 12
J = np.zeros((N, N))
for i in range(N - 1):
    J[i, i+1] = 1.0   # antiferromagnetic: anti-alignment preferred
    J[i+1, i] = 1.0

target_energy = -(N - 1)   # all bonds anti-aligned → E = -11

print(f'H₁₂ Heisenberg chain ({N}×{N} tridiagonal J-matrix)')
print(f'  Bond coupling J = +1  (antiferromagnetic)')
print(f'  Number of bonds : {N-1}')
print(f'  Target energy   : {target_energy}')
print(f'  J matrix (first 4×4):')
print(J[:4, :4])

## Solve with KLT Ground-State Engine

`client.klt.run()` submits the interaction matrix to Qumulator's KLT relaxation engine.
It returns the ground-state phases $\phi_i \in [0, 2\pi)$ per site, from which
we extract Ising spins via $s_i = \text{sign}(\cos \phi_i)$.

In [ ]:
t0     = time.perf_counter()
result = client.klt.run(
    interaction_matrix=J.tolist(),
    confinement_strength=0.05,
    cluster_size=2,
)
t_klt = time.perf_counter() - t0

phases = np.array(result.states)      # KLT phase per site
spins  = np.sign(np.cos(phases))      # ±1 Ising spin

# XY energy from KLT phases
E_xy    = float(result.energy)
# Ising binary energy E = -½ sᵀ J s  (spin-flip convention)
E_ising = float(-0.5 * spins @ J @ spins)
pct_err = abs(E_ising - target_energy) / abs(target_energy) * 100

spin_sym = ['↓' if s < 0 else '↑' for s in spins]
is_afm   = all(spin_sym[i] != spin_sym[i+1] for i in range(N-1))

print(f'KLT energy (XY)    : {E_xy:.6f}')
print(f'KLT energy (Ising) : {E_ising:.6f}')
print(f'Target energy      : {target_energy:.6f}')
print(f'Error              : {pct_err:.2f}%')
print(f'Execution time     : {t_klt:.3f} s')
print(f'Spin configuration : {" ".join(spin_sym)}')
print(f'Antiferromagnetic  : {"✓ YES" if is_afm else "✗ NO"}')

In [ ]:
print('\n' + '='*60)
print(' BENCHMARK: H₁₂ Hydrogen Chain Ground State (VQE Equivalent)')
print('='*60)
print(f'  KLT energy (Ising) : {E_ising:.4f}  (target: {target_energy})')
print(f'  Error              : {pct_err:.2f}%')
print(f'  Spin config        : {" ".join(spin_sym)}')
print(f'  Néel order found   : {"✓ YES" if is_afm else "✗ NO"}')

pass_ = pct_err < 10.0
print(f'\n  Result: {"✓ PASS" if pass_ else "✗ FAIL"} ({pct_err:.1f}% from target)')
assert pass_, f'h12 VQE failed: {pct_err:.1f}% error'